# Lab Day 2 — DeepWeeds (Nguyễn Hoàng Sơn, 2A202602457)

Notebook chạy trên **Google Colab (GPU T4)**. Code nằm ở `submissions/2A202602457_NguyenHoangSon/code/`
(`dataset.py`, `model.py`, `losses.py`, `train.py`, `inference.py`, `benchmark.py`), mọi thí nghiệm đi qua
**một** hàm `train.run(Config(...))`.

- Ảnh giải nén vào đĩa cục bộ `/content/data` (nhanh); **mọi kết quả** (`runs/`, `predictions/`, `curves/`)
  ghi vào Google Drive `MyDrive/K4_DAY02` để không mất khi Colab ngắt. Chạy lại ô nào cũng an toàn:
  thí nghiệm đã xong (có `summary.json`) được bỏ qua, thí nghiệm dở dang được **resume** từ epoch cuối.
- Quy tắc: chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

## 0. Cài đặt môi trường

In [ ]:
import os, sys, json, glob, time, platform
USE_DRIVE = True
REPO_URL = "https://github.com/sown101/K4-DAY02-NguyenHoangSon-2A202602457.git"
REPO = "/content/K4-DAY02-NguyenHoangSon-2A202602457"
if not os.path.exists(REPO):
    !git clone -q {REPO_URL} {REPO}
else:
    !git -C {REPO} pull -q
CODE = f"{REPO}/submissions/2A202602457_NguyenHoangSon/code"
for p in (REPO, CODE):
    if p not in sys.path:
        sys.path.insert(0, p)
!pip -q install timm openpyxl

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/K4_DAY02"
else:
    OUT = "/content/out"
os.makedirs(OUT, exist_ok=True)

import numpy as np, pandas as pd, torch, torchvision, timm
print("python", platform.python_version(), "| torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU - Runtime > Change runtime type > T4 GPU")
print("OUT =", OUT)

### Tải dữ liệu
Ảnh từ Zenodo (~490 MB, kiểm tra MD5 `b7b30f96d466fba86016aa5a26606e0f`), nhãn và file chia fold từ GitHub
của tác giả (không sửa). Bản `images.zip` được lưu thêm vào Drive để lần sau chép lại nhanh.

In [ ]:
import hashlib, shutil
DATA = "/content/data"
os.makedirs(f"{DATA}/labels", exist_ok=True)
ZIP, ZIP_DRIVE = f"{DATA}/images.zip", f"{OUT}/images.zip"
if not os.path.exists(ZIP):
    if os.path.exists(ZIP_DRIVE):
        shutil.copy(ZIP_DRIVE, ZIP)
    else:
        !wget -q -O {ZIP} "https://zenodo.org/records/7939060/files/images.zip?download=1"
h = hashlib.md5()
with open(ZIP, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"
if USE_DRIVE and not os.path.exists(ZIP_DRIVE):
    shutil.copy(ZIP, ZIP_DRIVE)
!unzip -q -n {ZIP} -d {DATA}/images_raw

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    if not os.path.exists(f"{DATA}/labels/{name}.csv"):
        !wget -q -O {DATA}/labels/{name}.csv {BASE}/{name}.csv

# thư mục chứa nhiều .jpg nhất là thư mục ảnh
dirs = {}
for p in glob.glob(f"{DATA}/images_raw/**/*.jpg", recursive=True):
    dirs[os.path.dirname(p)] = dirs.get(os.path.dirname(p), 0) + 1
IMAGES_DIR = max(dirs, key=dirs.get)
LABELS_DIR = f"{DATA}/labels"
print("IMAGES_DIR =", IMAGES_DIR, "| số ảnh:", dirs[IMAGES_DIR])

## Bước 0 — EDA và kiểm tra chia dữ liệu (README mục 2.1)

In [ ]:
import dataset as ds
import matplotlib.pyplot as plt
EDA = f"{OUT}/eda"; os.makedirs(EDA, exist_ok=True)
train_df, val_df, test_df = ds.load_split(LABELS_DIR, fold=0)
split_report = ds.check_split(train_df, val_df, test_df, IMAGES_DIR)
json.dump(split_report, open(f"{EDA}/split_report.json", "w"), indent=2, ensure_ascii=False)

paper = [1125, 1064, 1031, 1022, 1062, 1009, 1074, 1016, 9106]   # Table 1 của bài báo
pc = pd.DataFrame(split_report["per_class"]).T
pc["paper_table1"] = paper
pc["chênh_lệch"] = pc["total"] - pc["paper_table1"]
display(pc)
print("Tỉ lệ lớp lớn nhất / nhỏ nhất (train):", round(pc["train"].max() / pc["train"].min(), 2))

ax = pc[["train", "val", "test"]].plot.bar(figsize=(10, 4), title="DeepWeeds fold 0: số ảnh mỗi lớp")
ax.set_ylabel("số ảnh"); plt.tight_layout(); plt.savefig(f"{EDA}/class_distribution.png", dpi=130); plt.show()

In [ ]:
# >= 3 ảnh mỗi lớp
from PIL import Image
fig, axes = plt.subplots(9, 4, figsize=(9, 20))
for c in range(9):
    files = train_df[train_df.Label == c].sample(4, random_state=0).Filename
    for j, fn in enumerate(files):
        axes[c, j].imshow(Image.open(f"{IMAGES_DIR}/{fn}")); axes[c, j].axis("off")
        if j == 0: axes[c, j].set_title(ds.CLASS_NAMES[c], loc="left", fontsize=10)
plt.tight_layout(); plt.savefig(f"{EDA}/samples_per_class.png", dpi=90); plt.show()
im = Image.open(f"{IMAGES_DIR}/{train_df.Filename.iloc[0]}"); print("kích thước, mode ảnh:", im.size, im.mode)

## Bước 0 — Kiểm tra pipeline (slide trang 59, GUIDE mục 1.3)
1) seed cố định · 2) loss ban đầu ≈ ln 9 = 2.197 · 3) overfit một batch nhỏ · 4) ảnh sau augmentation khớp nhãn ·
5) unit test các phần dễ sai (`tests_code.py`: focal γ=0 ≡ CE, CutMix, gộp BN, temperature, EMA...).

In [ ]:
!cd {CODE} && python -m unittest tests_code 2>&1 | tail -3

In [ ]:
import torch.nn.functional as F
import model as mdl, train as tr
tr.set_seed(0)
dev = "cuda"
m = mdl.build_model("resnet50").to(dev)
mean, std = m.pretrained_cfg["mean"], m.pretrained_cfg["std"]
tf = ds.build_transforms(True, 224, "basic", mean, std)
loader = ds.make_loader(train_df, IMAGES_DIR, tf, 32, train=True, num_workers=2, seed=0)
x, y, _ = next(iter(loader)); x, y = x.to(dev), torch.as_tensor(y).to(dev)
m.eval()
with torch.no_grad():
    l0 = F.cross_entropy(m(x), y).item()
print(f"loss ban đầu = {l0:.4f}  (kỳ vọng ≈ {np.log(9):.4f})")

# overfit 8 ảnh
xb, yb = x[:8], y[:8]
opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
m.train()
for i in range(60):
    loss = F.cross_entropy(m(xb), yb); opt.zero_grad(); loss.backward(); opt.step()
    if i % 10 == 0 or i == 59: print(f"bước {i:2d} loss {loss.item():.4f}")
del m, opt; torch.cuda.empty_cache()

In [ ]:
# ảnh sau augmentation (đã giải chuẩn hoá) + nhãn, và CutMix
import losses
fig, axes = plt.subplots(2, 6, figsize=(15, 5.5))
for i in range(6):
    axes[0, i].imshow(ds.denormalize(x[i].cpu(), mean, std).permute(1, 2, 0)); axes[0, i].axis("off")
    axes[0, i].set_title(ds.CLASS_NAMES[int(y[i])], fontsize=9)
xm, (ya, yb2, lam) = losses.mix_batch(x[:6].cpu(), y[:6].cpu(), 1.0, "cutmix", np.random.default_rng(0))
for i in range(6):
    axes[1, i].imshow(ds.denormalize(xm[i], mean, std).permute(1, 2, 0)); axes[1, i].axis("off")
    axes[1, i].set_title(f"{ds.CLASS_NAMES[int(ya[i])][:10]} / {ds.CLASS_NAMES[int(yb2[i])][:10]}\nlam={lam:.2f}", fontsize=8)
plt.suptitle("Hàng 1: augmentation basic · Hàng 2: CutMix (nhãn a / nhãn b, lam theo diện tích thật)")
plt.tight_layout(); plt.savefig(f"{EDA}/augmentation_check.png", dpi=100); plt.show()

## Hàm chạy chung
`run_exp(...)` = `train.run(Config(...))` với đường dẫn chuẩn; bỏ qua nếu đã xong, resume nếu dở dang.

In [ ]:
import importlib
importlib.reload(tr)
from train import Config, run
COMMON = dict(images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, out_dir=f"{OUT}/runs",
              pred_dir=f"{OUT}/predictions", curves_dir=f"{OUT}/curves", num_workers=2)

def run_exp(**kw):
    cfg = Config(**{**COMMON, **kw})
    done = tr.run_dir(cfg) / "summary.json"
    if done.exists():
        print(f"bỏ qua {cfg.exp_id} seed{cfg.seed}: đã xong")
        return json.load(open(done))
    return run(cfg)

def results(prefix=""):
    rows = [json.load(open(p)) for p in sorted(glob.glob(f"{OUT}/runs/{prefix}*/seed*/summary.json"))]
    if not rows: return pd.DataFrame()
    cols = ["exp_id", "seed", "backbone", "desc", "params_M", "gmacs", "best_epoch", "val_macro_f1", "val_top1",
            "val_f1_chinee", "val_f1_snake", "val_ece", "train_s_per_epoch", "latency_b1_p50_ms"]
    return pd.DataFrame(rows)[cols].round(4)

## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (AdamW, LR backbone 1e-4 / head 1e-3, wd 0.05 trừ norm/bias, warmup 1 epoch + cosine,
CE, batch 64, **12 epoch**, AMP, 224 px, chọn checkpoint theo macro-F1 val), **cùng seed 0**.
Ràng buộc: ResNet (B01) · ConvNeXt (B02) · transformer DeiT-S (B03) · mạng nhẹ EfficientNet-B0 (B04), MobileNetV3-L (B05).

In [ ]:
BACKBONES = [("B01", "resnet50"), ("B02", "convnext_tiny"), ("B03", "deit_small"),
             ("B04", "efficientnet_b0"), ("B05", "mobilenetv3")]
for exp_id, bb in BACKBONES:
    run_exp(exp_id=exp_id, backbone=bb, desc=bb, seed=0)
    torch.cuda.empty_cache()
results("B")

## Bước 2 — Công thức huấn luyện (≥ 3 trục, mỗi lần khác `T00` đúng một yếu tố)
Đặt `BB` = backbone chọn ở Bước 1 (lý do dựa trên số liệu ghi vào báo cáo). `T00` = công thức nền trên `BB`.

In [ ]:
BB = "resnet50"   # TODO: đổi theo kết quả Bước 1
ABLATIONS = [
    ("T00", "baseline", {}),
    # A. khởi tạo
    ("T01", "init_scratch", dict(init="scratch")),
    ("T02", "init_frozen", dict(init="frozen")),
    # C. loss
    ("T03", "loss_ls0.1", dict(loss="ls", label_smoothing=0.1)),
    ("T04", "loss_focal_g2", dict(loss="focal", focal_gamma=2.0)),
    ("T05", "loss_ce_weighted", dict(loss="ce_weighted")),
    # B. augmentation
    ("T06", "aug_color", dict(aug="color")),
    ("T07", "aug_trivial", dict(aug="trivial")),
    ("T08", "aug_cutmix", dict(mix="cutmix", mix_alpha=1.0)),
    # D. cân bằng mẫu
    ("T09", "sampler_balanced", dict(sampler="balanced")),
    # F. chính quy hoá
    ("T10", "ema0.998", dict(ema_decay=0.998)),
]
for exp_id, desc, kw in ABLATIONS:
    run_exp(exp_id=exp_id, backbone=BB, desc=desc, seed=0, **kw)
    torch.cuda.empty_cache()
r = results("T")
base = r.loc[r.exp_id == "T00", "val_macro_f1"].iloc[0]
r["delta_vs_T00"] = (r["val_macro_f1"] - base).round(4)
r

In [ ]:
# Kết hợp các yếu tố thắng rõ rệt (TODO: chọn theo bảng trên, ghi rõ là "tham lam theo trục")
# run_exp(exp_id="T11", backbone=BB, desc="combo_...", seed=0, **{...})

## Bước 3, 4, 5
Sẽ bổ sung sau khi có kết quả Bước 1–2: suy luận (TTA, multi-scale, độ phân giải, prob vs logit, ensemble,
EMA, temperature scaling, gộp BN/FP16, độ trễ), chung kết ≥ 3 seed + mốc `T00`, `eval.py score/grade`, `results.xlsx`.